# Feature Visualization Video

指定した元動画に特徴量レイヤーを重ねた確認用動画を生成します。

現時点では以下を描画できます。

- グリッド線
- 各グリッドの optical flow 平均ベクトルを矢印で表示

`RENDER_LAYERS` と `FEATURE_OUTPUTS` を切り替えることで、今後ほかの特徴量描画を追加しやすい構成にしています。

In [7]:
from __future__ import annotations

from pathlib import Path
import shutil
import subprocess
from typing import Any

import cv2
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    class tqdm:
        def __init__(self, total=None, desc=None):
            self.total = total
            self.desc = desc
        def __enter__(self):
            return self
        def __exit__(self, exc_type, exc, tb):
            return False
        def update(self, n=1):
            pass


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
MOVIE_PREPROCESS_DIR = PROJECT_ROOT / "data" / "movie_preprocess"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "feature_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ここで可視化したい動画を複数指定します。
# data/movie_preprocess 配下から {sample_id}_{front|rear}.mp4 を探します。
# view は "front" / "rear"、views は ["front", "rear"] のように複数指定できます。
VIDEO_TARGETS = [
    # {"sample_id": "001_後進時にトラックに衝突", "view": "front"},
    # {"sample_id": "001_後進時にトラックに衝突", "view": "rear"},
    # {"sample_id": "005_後進旋回時トラックに衝突", "view": "front"},
    # {"sample_id": "005_後進旋回時トラックに衝突", "view": "rear"},
    # {"sample_id": "006_安全ポールに接触", "view": "front"},
    # {"sample_id": "006_安全ポールに接触", "view": "rear"},
    # {"sample_id": "012_安全ポール乗り上げ", "view": "front"},
    # {"sample_id": "012_安全ポール乗り上げ", "view": "rear"},
    {"sample_id": "090_前進時、電源ケーブル乗り上げ", "view": "front"},
    {"sample_id": "090_前進時、電源ケーブル乗り上げ", "view": "rear"},
    {"sample_id": "1001", "view": "front"},
    {"sample_id": "1001", "view": "rear"},
    # {"sample_id": "1038", "view": "front"},
    # {"sample_id": "1038", "view": "rear" },
    {"sample_id": "1002", "view": "front"},
    {"sample_id": "1002", "view": "rear"},
    {"sample_id": "1006", "view": "front"},
    {"sample_id": "1006", "view": "rear"},
    {"sample_id": "1024", "view": "front"},
    {"sample_id": "1024", "view": "rear"},
    {"sample_id": "1036", "view": "front"},
    {"sample_id": "1036", "view": "rear"},
]


def find_movie_preprocess_video(
    sample_id: str,
    view: str,
    movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR,
) -> Path:
    view = view.lower().strip()
    if view not in {"front", "rear"}:
        raise ValueError(f"view must be 'front' or 'rear': {view}")
    filename = f"{sample_id}_{view}.mp4"
    candidates = sorted(movie_preprocess_dir.glob(f"*/*/{filename}"))
    candidates = [path for path in candidates if path.is_file()]
    if not candidates:
        raise FileNotFoundError(
            f"Could not find {filename} under {movie_preprocess_dir}"
        )
    if len(candidates) > 1:
        raise ValueError("Multiple matching videos were found. Please make sample_id unique or narrow the search logic:\n" + "\n".join(str(p) for p in candidates))
    return candidates[0]


def expand_video_targets(video_targets: list[dict[str, Any]]) -> list[dict[str, str]]:
    expanded = []
    for target in video_targets:
        sample_id = str(target["sample_id"])
        if "views" in target:
            views = target["views"]
        else:
            views = [target.get("view", "front")]
        for view in views:
            expanded.append({
                "sample_id": sample_id,
                "view": str(view).lower().strip(),
            })
    if not expanded:
        raise ValueError("VIDEO_TARGETS must contain at least one target.")
    return expanded


def build_output_paths(input_video_path: Path, sample_id: str, view: str) -> tuple[Path, Path]:
    try:
        category, environment = input_video_path.relative_to(MOVIE_PREPROCESS_DIR).parts[:2]
    except ValueError:
        category, environment = "unknown", "unknown"
    output_stem = f"{sample_id}_{view}_{category}_{environment}_grid_flow"
    return OUTPUT_DIR / f"{output_stem}.mp4", OUTPUT_DIR / f"{output_stem}_features.csv"


EXPANDED_VIDEO_TARGETS = expand_video_targets(VIDEO_TARGETS)

# 描画レイヤーの ON/OFF。今後の特徴量描画はここへキーを増やす想定です。
RENDER_LAYERS = {
    "grid": True,
    "grid_flow_arrows": True,
    "grid_flow_labels": False,
}

# CSV に保存する特徴量の ON/OFF。動画だけ欲しい場合は False にできます。
FEATURE_OUTPUTS = {
    "grid_flow": True,
}

VISUALIZATION_CONFIG: dict[str, Any] = {
    "resize_width": 960,
    "frame_stride": 1,
    "max_frames": None,
    "fourcc": "mp4v",
    "browser_compatible_mp4": True,
    "keep_intermediate_video": False,
    "ffmpeg_video_codec": "libx264",
    "ffmpeg_crf": 20,
    "ffmpeg_preset": "medium",
    "grid": {
        "shape": (3, 3),
        "line_color": (80, 220, 255),
        "line_thickness": 1,
    },
    "flow_arrows": {
        "color": (40, 255, 80),
        "tip_color": (0, 180, 255),
        "outline_color": (0, 0, 0),
        "outline_extra_thickness": 4,
        "thickness": 2,
        "tip_length": 0.25,
        "scale": 12.0,
        "max_length_ratio": 0.42,
        "min_magnitude": 0.02,
    },
    "flow_labels": {
        "font_scale": 0.42,
        "color": (255, 255, 255),
        "shadow_color": (0, 0, 0),
        "thickness": 1,
    },
    "farneback": {
        "pyr_scale": 0.5,
        "levels": 3,
        "winsize": 15,
        "iterations": 3,
        "poly_n": 5,
        "poly_sigma": 1.2,
        "flags": 0,
    },
}

print(f"PROJECT_ROOT      : {PROJECT_ROOT}")
print(f"MOVIE_PREPROCESS  : {MOVIE_PREPROCESS_DIR}")
print(f"video targets     : {len(EXPANDED_VIDEO_TARGETS)}")
for target in EXPANDED_VIDEO_TARGETS:
    print(f"- {target['sample_id']} / {target['view']}")
print(f"RENDER_LAYERS     : {RENDER_LAYERS}")


PROJECT_ROOT      : /workspaces/forklift
MOVIE_PREPROCESS  : /workspaces/forklift/data/movie_preprocess
video targets     : 4
- 006_安全ポールに接触 / front
- 006_安全ポールに接触 / rear
- 012_安全ポール乗り上げ / front
- 012_安全ポール乗り上げ / rear
RENDER_LAYERS     : {'grid': True, 'grid_flow_arrows': True, 'grid_flow_labels': False}


In [9]:
def resize_keep_aspect(frame: np.ndarray, width: int | None) -> np.ndarray:
    if width is None or frame.shape[1] <= width:
        return frame
    scale = width / frame.shape[1]
    height = max(1, int(round(frame.shape[0] * scale)))
    return cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)


def open_video(video_path: str | Path) -> cv2.VideoCapture:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")
    return capture


def make_video_writer(output_path: str | Path, fps: float, frame_shape: tuple[int, ...], fourcc_name: str = "mp4v") -> cv2.VideoWriter:
    h, w = frame_shape[:2]
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*fourcc_name),
        float(fps),
        (int(w), int(h)),
    )
    if not writer.isOpened():
        raise RuntimeError(f"Could not create video writer: {output_path}")
    return writer


def build_intermediate_video_path(output_path: str | Path) -> Path:
    output_path = Path(output_path)
    return output_path.with_name(f"{output_path.stem}_opencv_tmp{output_path.suffix}")


def transcode_to_browser_compatible_mp4(input_path: str | Path, output_path: str | Path, config: dict[str, Any]) -> bool:
    input_path = Path(input_path)
    output_path = Path(output_path)
    ffmpeg_path = shutil.which("ffmpeg")
    if ffmpeg_path is None:
        print("ffmpeg was not found. Keeping the OpenCV-encoded mp4; VSCode may not preview it.")
        if input_path != output_path:
            input_path.replace(output_path)
        return False

    cmd = [
        ffmpeg_path,
        "-y",
        "-i", str(input_path),
        "-an",
        "-c:v", str(config.get("ffmpeg_video_codec", "libx264")),
        "-preset", str(config.get("ffmpeg_preset", "medium")),
        "-crf", str(config.get("ffmpeg_crf", 20)),
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(output_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("ffmpeg transcode failed. Keeping the OpenCV-encoded mp4; VSCode may not preview it.")
        print(result.stderr[-2000:])
        if input_path != output_path:
            input_path.replace(output_path)
        return False
    if input_path != output_path and not bool(config.get("keep_intermediate_video", False)):
        input_path.unlink(missing_ok=True)
    return True


def grid_bounds(frame_shape: tuple[int, ...], grid_shape: tuple[int, int]) -> list[dict[str, int]]:
    h, w = frame_shape[:2]
    gy_n, gx_n = grid_shape
    cells = []
    for gy in range(gy_n):
        y0, y1 = int(h * gy / gy_n), int(h * (gy + 1) / gy_n)
        for gx in range(gx_n):
            x0, x1 = int(w * gx / gx_n), int(w * (gx + 1) / gx_n)
            cells.append({"gy": gy, "gx": gx, "x0": x0, "y0": y0, "x1": x1, "y1": y1})
    return cells


def compute_grid_flow_features(flow: np.ndarray, frame_index: int, time_sec: float, grid_shape: tuple[int, int]) -> list[dict[str, float]]:
    fx = flow[..., 0]
    fy = flow[..., 1]
    mag = np.sqrt(fx * fx + fy * fy)
    rows = []
    for cell in grid_bounds(flow.shape[:2], grid_shape):
        x0, y0, x1, y1 = cell["x0"], cell["y0"], cell["x1"], cell["y1"]
        cell_fx = fx[y0:y1, x0:x1]
        cell_fy = fy[y0:y1, x0:x1]
        cell_mag = mag[y0:y1, x0:x1]
        mean_dx = float(np.mean(cell_fx)) if cell_fx.size else 0.0
        mean_dy = float(np.mean(cell_fy)) if cell_fy.size else 0.0
        rows.append({
            "frame_index": int(frame_index),
            "time_sec": float(time_sec),
            "grid_y": int(cell["gy"]),
            "grid_x": int(cell["gx"]),
            "x0": int(x0),
            "y0": int(y0),
            "x1": int(x1),
            "y1": int(y1),
            "center_x": float((x0 + x1) / 2),
            "center_y": float((y0 + y1) / 2),
            "flow_dx_mean": mean_dx,
            "flow_dy_mean": mean_dy,
            "flow_mag_mean": float(np.mean(cell_mag)) if cell_mag.size else 0.0,
            "flow_vector_mag": float(np.hypot(mean_dx, mean_dy)),
        })
    return rows


In [10]:
def draw_grid(frame: np.ndarray, grid_shape: tuple[int, int], config: dict[str, Any]) -> np.ndarray:
    color = tuple(config.get("line_color", (80, 220, 255)))
    thickness = int(config.get("line_thickness", 1))
    h, w = frame.shape[:2]
    gy_n, gx_n = grid_shape
    for gy in range(1, gy_n):
        y = int(h * gy / gy_n)
        cv2.line(frame, (0, y), (w - 1, y), color, thickness, lineType=cv2.LINE_AA)
    for gx in range(1, gx_n):
        x = int(w * gx / gx_n)
        cv2.line(frame, (x, 0), (x, h - 1), color, thickness, lineType=cv2.LINE_AA)
    return frame


def draw_grid_flow_arrows(frame: np.ndarray, grid_features: list[dict[str, float]], config: dict[str, Any]) -> np.ndarray:
    color = tuple(config.get("color", (40, 255, 80)))
    tip_color = tuple(config.get("tip_color", color))
    outline_color = tuple(config.get("outline_color", (0, 0, 0)))
    thickness = int(config.get("thickness", 2))
    outline_thickness = thickness + int(config.get("outline_extra_thickness", 4))
    tip_length = float(config.get("tip_length", 0.25))
    scale = float(config.get("scale", 12.0))
    max_length_ratio = float(config.get("max_length_ratio", 0.42))
    min_magnitude = float(config.get("min_magnitude", 0.02))
    for row in grid_features:
        vector_mag = float(row["flow_vector_mag"])
        if vector_mag < min_magnitude:
            continue
        cx, cy = float(row["center_x"]), float(row["center_y"])
        dx, dy = float(row["flow_dx_mean"]), float(row["flow_dy_mean"])
        cell_w = max(1.0, float(row["x1"] - row["x0"]))
        cell_h = max(1.0, float(row["y1"] - row["y0"]))
        max_len = min(cell_w, cell_h) * max_length_ratio
        arrow_dx = dx * scale
        arrow_dy = dy * scale
        arrow_len = float(np.hypot(arrow_dx, arrow_dy))
        if arrow_len > max_len:
            ratio = max_len / max(arrow_len, 1e-6)
            arrow_dx *= ratio
            arrow_dy *= ratio
        start = (int(round(cx - arrow_dx * 0.5)), int(round(cy - arrow_dy * 0.5)))
        end = (int(round(cx + arrow_dx * 0.5)), int(round(cy + arrow_dy * 0.5)))
        cv2.arrowedLine(frame, start, end, outline_color, outline_thickness, line_type=cv2.LINE_AA, tipLength=tip_length)
        cv2.arrowedLine(frame, start, end, color, thickness, line_type=cv2.LINE_AA, tipLength=tip_length)
        cv2.circle(frame, end, max(3, outline_thickness + 1), outline_color, -1, lineType=cv2.LINE_AA)
        cv2.circle(frame, end, max(2, thickness + 1), tip_color, -1, lineType=cv2.LINE_AA)
    return frame


def draw_grid_flow_labels(frame: np.ndarray, grid_features: list[dict[str, float]], config: dict[str, Any]) -> np.ndarray:
    font_scale = float(config.get("font_scale", 0.42))
    color = tuple(config.get("color", (255, 255, 255)))
    shadow_color = tuple(config.get("shadow_color", (0, 0, 0)))
    thickness = int(config.get("thickness", 1))
    for row in grid_features:
        text = f"{row['flow_mag_mean']:.2f}"
        x = int(row["x0"] + 6)
        y = int(row["y0"] + 18)
        cv2.putText(frame, text, (x + 1, y + 1), cv2.FONT_HERSHEY_SIMPLEX, font_scale, shadow_color, thickness + 2, cv2.LINE_AA)
        cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, thickness, cv2.LINE_AA)
    return frame


def render_feature_layers(
    frame: np.ndarray,
    grid_features: list[dict[str, float]],
    render_layers: dict[str, bool],
    config: dict[str, Any],
) -> np.ndarray:
    out = frame.copy()
    grid_shape = tuple(config["grid"]["shape"])
    if render_layers.get("grid", False):
        out = draw_grid(out, grid_shape, config["grid"])
    if render_layers.get("grid_flow_arrows", False):
        out = draw_grid_flow_arrows(out, grid_features, config["flow_arrows"])
    if render_layers.get("grid_flow_labels", False):
        out = draw_grid_flow_labels(out, grid_features, config["flow_labels"])
    return out


In [11]:
def create_feature_visualization_video(
    input_video_path: str | Path,
    output_video_path: str | Path,
    output_feature_csv_path: str | Path | None = None,
    render_layers: dict[str, bool] = RENDER_LAYERS,
    feature_outputs: dict[str, bool] = FEATURE_OUTPUTS,
    config: dict[str, Any] = VISUALIZATION_CONFIG,
) -> pd.DataFrame:
    input_video_path = Path(input_video_path)
    output_video_path = Path(output_video_path)
    final_output_video_path = output_video_path
    write_video_path = build_intermediate_video_path(output_video_path) if config.get("browser_compatible_mp4", True) else output_video_path
    capture = open_video(input_video_path)
    src_fps = capture.get(cv2.CAP_PROP_FPS)
    if not src_fps or np.isnan(src_fps) or src_fps <= 0:
        src_fps = 30.0
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    frame_stride = max(1, int(config.get("frame_stride", 1)))
    output_fps = src_fps / frame_stride
    max_frames = config.get("max_frames")

    ok, first_frame = capture.read()
    if not ok:
        capture.release()
        raise RuntimeError(f"No frames were read from {input_video_path}")
    first_frame = resize_keep_aspect(first_frame, config.get("resize_width"))
    writer = make_video_writer(write_video_path, output_fps, first_frame.shape, fourcc_name=config.get("fourcc", "mp4v"))

    grid_shape = tuple(config["grid"]["shape"])
    farneback = config["farneback"]
    feature_rows: list[dict[str, float]] = []

    prev_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
    first_features = [
        {
            **cell,
            "frame_index": 0,
            "time_sec": 0.0,
            "flow_dx_mean": 0.0,
            "flow_dy_mean": 0.0,
            "flow_mag_mean": 0.0,
            "flow_vector_mag": 0.0,
            "center_x": float((cell["x0"] + cell["x1"]) / 2),
            "center_y": float((cell["y0"] + cell["y1"]) / 2),
        }
        for cell in grid_bounds(first_frame.shape, grid_shape)
    ]
    writer.write(render_feature_layers(first_frame, first_features, render_layers, config))
    if feature_outputs.get("grid_flow", False):
        feature_rows.extend(first_features)

    written_frames = 1
    source_index = 1
    progress_total = frame_count if frame_count > 0 else None
    with tqdm(total=progress_total, desc="render visualization") as pbar:
        pbar.update(1)
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            if source_index % frame_stride != 0:
                source_index += 1
                pbar.update(1)
                continue
            if max_frames is not None and written_frames >= int(max_frames):
                break

            frame = resize_keep_aspect(frame, config.get("resize_width"))
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, **farneback)
            time_sec = source_index / src_fps
            grid_features = compute_grid_flow_features(flow, source_index, time_sec, grid_shape)
            writer.write(render_feature_layers(frame, grid_features, render_layers, config))
            if feature_outputs.get("grid_flow", False):
                feature_rows.extend(grid_features)

            prev_gray = gray
            source_index += 1
            written_frames += 1
            pbar.update(1)

    capture.release()
    writer.release()
    if config.get("browser_compatible_mp4", True):
        transcoded = transcode_to_browser_compatible_mp4(write_video_path, final_output_video_path, config)
        codec_label = "H.264/yuv420p" if transcoded else str(config.get("fourcc", "mp4v"))
    else:
        codec_label = str(config.get("fourcc", "mp4v"))

    features_df = pd.DataFrame(feature_rows)
    if output_feature_csv_path is not None and feature_outputs.get("grid_flow", False):
        output_feature_csv_path = Path(output_feature_csv_path)
        output_feature_csv_path.parent.mkdir(parents=True, exist_ok=True)
        features_df.to_csv(output_feature_csv_path, index=False)
    print(f"saved video   : {final_output_video_path} ({codec_label})")
    if output_feature_csv_path is not None and feature_outputs.get("grid_flow", False):
        print(f"saved features: {output_feature_csv_path}")
    print(f"written frames: {written_frames}")
    return features_df


In [12]:
batch_rows = []
feature_dfs = []

for target in EXPANDED_VIDEO_TARGETS:
    input_video_path = find_movie_preprocess_video(
        sample_id=str(target["sample_id"]),
        view=str(target["view"]),
    )
    output_video_path, output_feature_csv_path = build_output_paths(
        input_video_path=input_video_path,
        sample_id=str(target["sample_id"]),
        view=str(target["view"]),
    )
    print(f"\n=== {target['sample_id']} / {target['view']} ===")
    features_part = create_feature_visualization_video(
        input_video_path=input_video_path,
        output_video_path=output_video_path,
        output_feature_csv_path=output_feature_csv_path,
        render_layers=RENDER_LAYERS,
        feature_outputs=FEATURE_OUTPUTS,
        config=VISUALIZATION_CONFIG,
    )
    features_part.insert(0, "view", target["view"])
    features_part.insert(0, "sample_id", target["sample_id"])
    feature_dfs.append(features_part)
    batch_rows.append({
        "sample_id": target["sample_id"],
        "view": target["view"],
        "input_video_path": str(input_video_path),
        "output_video_path": str(output_video_path),
        "output_feature_csv_path": str(output_feature_csv_path),
        "feature_rows": len(features_part),
    })

batch_results_df = pd.DataFrame(batch_rows)
features_df = pd.concat(feature_dfs, ignore_index=True) if feature_dfs else pd.DataFrame()
display(batch_results_df)
display(features_df.head())



=== 006_安全ポールに接触 / front ===
saved video   : /workspaces/forklift/outputs/feature_visualization/006_安全ポールに接触_front_anomaly_indoor_grid_flow.mp4 (H.264/yuv420p)
saved features: /workspaces/forklift/outputs/feature_visualization/006_安全ポールに接触_front_anomaly_indoor_grid_flow_features.csv
written frames: 456

=== 006_安全ポールに接触 / rear ===
saved video   : /workspaces/forklift/outputs/feature_visualization/006_安全ポールに接触_rear_anomaly_indoor_grid_flow.mp4 (H.264/yuv420p)
saved features: /workspaces/forklift/outputs/feature_visualization/006_安全ポールに接触_rear_anomaly_indoor_grid_flow_features.csv
written frames: 456

=== 012_安全ポール乗り上げ / front ===
saved video   : /workspaces/forklift/outputs/feature_visualization/012_安全ポール乗り上げ_front_anomaly_indoor_grid_flow.mp4 (H.264/yuv420p)
saved features: /workspaces/forklift/outputs/feature_visualization/012_安全ポール乗り上げ_front_anomaly_indoor_grid_flow_features.csv
written frames: 445

=== 012_安全ポール乗り上げ / rear ===
saved video   : /workspaces/forklift/outputs/feature_vi

,sample_id,view,input_video_path,output_video_path,output_feature_csv_path,feature_rows
0,006_安全ポールに接触,front,/workspaces/forklift/data/movie_preprocess/ano...,/workspaces/forklift/outputs/feature_visualiza...,/workspaces/forklift/outputs/feature_visualiza...,4104
1,006_安全ポールに接触,rear,/workspaces/forklift/data/movie_preprocess/ano...,/workspaces/forklift/outputs/feature_visualiza...,/workspaces/forklift/outputs/feature_visualiza...,4104
2,012_安全ポール乗り上げ,front,/workspaces/forklift/data/movie_preprocess/ano...,/workspaces/forklift/outputs/feature_visualiza...,/workspaces/forklift/outputs/feature_visualiza...,4005
3,012_安全ポール乗り上げ,rear,/workspaces/forklift/data/movie_preprocess/ano...,/workspaces/forklift/outputs/feature_visualiza...,/workspaces/forklift/outputs/feature_visualiza...,4005


,sample_id,view,gy,gx,x0,y0,x1,y1,frame_index,time_sec,flow_dx_mean,flow_dy_mean,flow_mag_mean,flow_vector_mag,center_x,center_y,grid_y,grid_x
0,006_安全ポールに接触,front,0.0,0.0,0,0,320,180,0,0.0,0.0,0.0,0.0,0.0,160.0,90.0,NaN,NaN
1,006_安全ポールに接触,front,0.0,1.0,320,0,640,180,0,0.0,0.0,0.0,0.0,0.0,480.0,90.0,NaN,NaN
2,006_安全ポールに接触,front,0.0,2.0,640,0,960,180,0,0.0,0.0,0.0,0.0,0.0,800.0,90.0,NaN,NaN
3,006_安全ポールに接触,front,1.0,0.0,0,180,320,360,0,0.0,0.0,0.0,0.0,0.0,160.0,270.0,NaN,NaN
4,006_安全ポールに接触,front,1.0,1.0,320,180,640,360,0,0.0,0.0,0.0,0.0,0.0,480.0,270.0,NaN,NaN
